In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [3]:
documents = documents_llm

In [4]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
import json

user_prompt = json.dumps(doc)

In [9]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [10]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [11]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [12]:
result = response.output_parsed

print(result)

questions=['Can I still join the course if I just found it now?', 'Is it too late to start this course after the official start date?', 'If I join late, am I still eligible for a certificate?', 'What do I need to do to get the certificate if I’m joining now?', 'Does the course allow late participation, or do I have to wait for the next run?']


In [13]:
print(result.questions)

['Can I still join the course if I just found it now?', 'Is it too late to start this course after the official start date?', 'If I join late, am I still eligible for a certificate?', 'What do I need to do to get the certificate if I’m joining now?', 'Does the course allow late participation, or do I have to wait for the next run?']


In [14]:
from evaluation_utils import llm_structured

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course a bit late — is it still okay to join now?', 'Can I enroll after the course has already started, or is it too late?', 'If I join the course now, am I still eligible for a certificate?', "What do I need to do to get a certificate if I'm joining late?", 'Is it possible to submit the project late and still get certified?']


In [15]:
usage.input_tokens, usage.output_tokens

(207, 92)

In [16]:
from evaluation_utils import calc_price

cost = calc_price(usage)

cost

{'input_cost': 0.00015525, 'output_cost': 0.000414, 'total_cost': 0.00056925}

In [17]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course a bit late — is it still okay to join now?',
  'document': '74eb249bbf'},
 {'question': 'Can I enroll after the course has already started, or is it too late?',
  'document': '74eb249bbf'},
 {'question': 'If I join the course now, am I still eligible for a certificate?',
  'document': '74eb249bbf'},
 {'question': "What do I need to do to get a certificate if I'm joining late?",
  'document': '74eb249bbf'},
 {'question': 'Is it possible to submit the project late and still get certified?',
  'document': '74eb249bbf'}]

In [18]:
from evaluation_utils import llm_structured_retry

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [19]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [20]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

  0%|          | 0/113 [00:00<?, ?it/s]

565

In [21]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.08832300000000001

In [22]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.08832300000000001

In [23]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [25]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)

In [26]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [27]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [28]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [29]:
q = ground_truth[0]
q

{'question': 'Can I still join the course if I just found it now?',
 'document': '74eb249bbf'}

In [ ]:
doc_id = q["document"]
results = text_search(query=q["question"])

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'a9353fadfe',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The homework submission form is still open even though the deadline has passed — can I still submit?',
  'answer': "Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit."},
 {'id': '9f689c185f',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I missed the first homework - can I still get a certificate?',
  'answer': 'Yes, you need to pass the Capstone proj

In [32]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
a9353fadfe == 74eb249bbf: False
9f689c185f == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
69d122f12e == 74eb249bbf: False


In [33]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [34]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [35]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Can I still join the course if I just found it now?


[1, 0, 0, 0, 0]

In [36]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Where can I find the live stream URL for an Office Hours session before it starts?


[1, 0, 0, 0, 0]

In [37]:
q = ground_truth[50]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Where do I check the LLM Zoomcamp syllabus and current course structure?


[1, 0, 0, 0, 0]

In [38]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [39]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [40]:
relevance_total_text

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [41]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [42]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [43]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [44]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/565 [00:00<?, ?it/s]

In [55]:
cnt = 0

for line in relevance_total_text:
    if 1 in line:
        cnt = cnt + 1

cnt / len(ground_truth_sample)

0.8666666666666667

In [49]:
cnt = 0

for line in relevance_total:
    if 1 in line:
        cnt = cnt + 1

cnt / len(ground_truth)

0.8389380530973451

In [65]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    print(f"Hit count: {cnt}")        
    return cnt / len(relevance)


In [66]:
res = hit_rate(relevance_total_text)
print(res)


Hit count: 13
0.8666666666666667


In [ ]:
# Rank-based metric: Discounted Cumulative Gain (DCG)
# pos 1:  1,0
# pos 2:  0,5
# pos 3:  0,333
# not found:  0

total_score = 0.0

for line in relevance_total_text:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score = total_score + 1 / (rank + 1)
            break

total_score/len(relevance_total_text)

0.7688888888888888

In [70]:
total_score = 0.0

for line in relevance_total:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score = total_score + 1 / (rank + 1)
            break

total_score/len(relevance_total)

0.7345427728613567

In [71]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [74]:
print(mrr(relevance_total_text))
print(mrr(relevance_total))

0.7688888888888888
0.7345427728613567


In [75]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [76]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/565 [00:00<?, ?it/s]

Hit count: 474


{'hit_rate': 0.8389380530973451, 'mrr': 0.7345427728613567}